In [1]:
# some imports 
import numpy as np 
import networkx as nx
import copy 
from itertools import product
from collections import Counter
import matplotlib.pyplot as plt
from numpy import random
#D-Wave Ocean SDK related
import dimod
from dwave.embedding import embed_ising, unembed_sampleset
from minorminer import find_embedding
import pickle
from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # write files without needing a display


seed = 1224
random.seed(seed)

In [2]:
embeddings = pickle.loads(Path("../pkls/embeddings.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("../pkls/SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("../pkls/QA_sample_sets.pkl").read_bytes())
sa_results = pickle.loads(Path("../pkls/sa_results.pkl").read_bytes())
qa_results = pickle.loads(Path("../pkls/qa_results.pkl").read_bytes())
graphs = pickle.loads(Path("../pkls/graphs.pkl").read_bytes())
pt4_results = pickle.loads(Path("../pkls/pt4_results.pkl").read_bytes())

In [3]:
labels = ["Uniform Regular","Normal Regular","Binary Regular",
          "Uniform Erdos-Renyi","Normal Erdos-Renyi","Binary Erdos-Renyi",
          "Uniform Full","Normal Full","Binary Full",
          "Uniform Square","Normal Square","Binary Square",
          "Uniform Barabasi-Albert (1)","Normal Barabasi-Albert (1)","Binary Barabasi-Albert (1)",
          "Uniform Barabasi-Albert (2)","Normal Barabasi-Albert (2)","Binary Barabasi-Albert (2)"]

In [4]:
def print_latex_table(sa, qa, labels, caption="Simulated vs. quantum annealing results.",
                      label="tab:sa-vs-qa", precision=2, row_header="Instance"):
    """
    sa, qa : lists of (best, average) pairs, e.g. [(np.float64(-774.2), np.float64(-768.9)), ...]
    labels : list of row labels, one per pair, in the same order as sa and qa
    Prints a LaTeX table with one row per label.
    """
    if not (len(sa) == len(qa) == len(labels)):
        raise ValueError(
            f"Length mismatch: sa={len(sa)}, qa={len(qa)}, labels={len(labels)}"
        )

    fmt = f"{{:.{precision}f}}"

    def escape(s):
        # Escape LaTeX special characters in labels
        for ch, rep in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"),
                        ("$", r"\$"), ("#", r"\#"), ("_", r"\_"),
                        ("{", r"\{"), ("}", r"\}")]:
            s = s.replace(ch, rep)
        return s

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \begin{tabular}{l rr rr}")
    lines.append(r"        \toprule")
    lines.append(r"        & \multicolumn{2}{c}{\textbf{Simulated Annealing}} "
                 r"& \multicolumn{2}{c}{\textbf{Quantum Annealing}} \\")
    lines.append(r"        \cmidrule(lr){2-3} \cmidrule(lr){4-5}")
    lines.append(rf"        \textbf{{{row_header}}} & \textbf{{Lowest}} & \textbf{{Average}} "
                 r"& \textbf{Lowest} & \textbf{Average} \\")
    lines.append(r"        \midrule")

    for lab, (sa_best, sa_avg), (qa_best, qa_avg) in zip(labels, sa, qa):
        lines.append(
            f"        {escape(str(lab))} & "
            f"{fmt.format(float(sa_best))} & {fmt.format(float(sa_avg))} & "
            f"{fmt.format(float(qa_best))} & {fmt.format(float(qa_avg))} \\\\"
        )

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")

    print("\n".join(lines))

print_latex_table(sa_results, qa_results, labels)

\begin{table}[H]
    \centering
    \caption{Simulated vs. quantum annealing results.}
    \label{tab:sa-vs-qa}
    \begin{tabular}{l rr rr}
        \toprule
        & \multicolumn{2}{c}{\textbf{Simulated Annealing}} & \multicolumn{2}{c}{\textbf{Quantum Annealing}} \\
        \cmidrule(lr){2-3} \cmidrule(lr){4-5}
        \textbf{Instance} & \textbf{Lowest} & \textbf{Average} & \textbf{Lowest} & \textbf{Average} \\
        \midrule
        Uniform Regular & -774.21 & -768.87 & -744.01 & -723.85 \\
        Normal Regular & -1311.06 & -1297.66 & -1200.39 & -1137.64 \\
        Binary Regular & -1344.00 & -1343.58 & -1320.00 & -1294.32 \\
        Uniform Erdos-Renyi & -688.35 & -681.63 & -585.83 & -468.44 \\
        Normal Erdos-Renyi & -1205.84 & -1183.17 & -1072.18 & -849.30 \\
        Binary Erdos-Renyi & -1235.00 & -1219.22 & -1071.00 & -853.17 \\
        Uniform Full & -984.02 & -971.09 & -831.94 & -674.82 \\
        Normal Full & -1684.68 & -1662.71 & -1485.08 & -1170.16 \\
        Bi

In [5]:
def _escape(s):
    for ch, rep in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"),
                    ("$", r"\$"), ("#", r"\#"), ("_", r"\_"),
                    ("{", r"\{"), ("}", r"\}")]:
        s = s.replace(ch, rep)
    return s


def print_latex_diff_table(sa, qa, labels,
                           caption="Differences between simulated and quantum annealing, sorted by lowest-energy difference.",
                           label="tab:sa-qa-diff", precision=2, row_header="Instance",
                           sort_by="d_best"):
    """
    sa, qa : lists of (best, average) pairs
    labels : list of row labels, same order as sa and qa
    sort_by: which column orders the rows from high to low:
             "d_best"  (SA - QA, Lowest)
             "d_avg"   (SA - QA, Average)
             "gap_sa"  (SA Average - Lowest)
             "gap_qa"  (QA Average - Lowest)

    Columns:
      SA - QA (Lowest)       = sa_best - qa_best
      SA - QA (Average)      = sa_avg  - qa_avg
      SA gap (Avg - Lowest)  = sa_avg  - sa_best
      QA gap (Avg - Lowest)  = qa_avg  - qa_best
    """
    if not (len(sa) == len(qa) == len(labels)):
        raise ValueError(
            f"Length mismatch: sa={len(sa)}, qa={len(qa)}, labels={len(labels)}"
        )
    if sort_by not in ("d_best", "d_avg", "gap_sa", "gap_qa"):
        raise ValueError(f"Unknown sort_by: {sort_by!r}")

    def f(x):
        # Math mode so negative values get a proper minus sign
        return f"${float(x):.{precision}f}$"

    rows = []
    for lab, (sa_best, sa_avg), (qa_best, qa_avg) in zip(labels, sa, qa):
        rows.append({
            "label": str(lab),
            "d_best": float(sa_best) - float(qa_best),
            "d_avg":  float(sa_avg) - float(qa_avg),
            "gap_sa": float(sa_avg) - float(sa_best),
            "gap_qa": float(qa_avg) - float(qa_best),
        })

    # High to low; sorted() is stable, so ties keep the original label order
    rows = sorted(rows, key=lambda r: r[sort_by], reverse=True)

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \begin{tabular}{l rr rr}")
    lines.append(r"        \toprule")
    lines.append(r"        & \multicolumn{2}{c}{\textbf{SA $-$ QA}} "
                 r"& \multicolumn{2}{c}{\textbf{Average $-$ Lowest}} \\")
    lines.append(r"        \cmidrule(lr){2-3} \cmidrule(lr){4-5}")
    lines.append(rf"        \textbf{{{row_header}}} & \textbf{{Lowest}} & \textbf{{Average}} "
                 r"& \textbf{SA} & \textbf{QA} \\")
    lines.append(r"        \midrule")

    for r in rows:
        lines.append(
            f"        {_escape(r['label'])} & {f(r['d_best'])} & {f(r['d_avg'])} & "
            f"{f(r['gap_sa'])} & {f(r['gap_qa'])} \\\\"
        )

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")

    print("\n".join(lines))


print_latex_diff_table(sa_results, qa_results, labels)                    # sorted by SA - QA (Lowest)
# print_latex_diff_table(sa_results, qa_results, labels, sort_by="d_avg")  # or by another column

\begin{table}[H]
    \centering
    \caption{Differences between simulated and quantum annealing, sorted by lowest-energy difference.}
    \label{tab:sa-qa-diff}
    \begin{tabular}{l rr rr}
        \toprule
        & \multicolumn{2}{c}{\textbf{SA $-$ QA}} & \multicolumn{2}{c}{\textbf{Average $-$ Lowest}} \\
        \cmidrule(lr){2-3} \cmidrule(lr){4-5}
        \textbf{Instance} & \textbf{Lowest} & \textbf{Average} & \textbf{SA} & \textbf{QA} \\
        \midrule
        Binary Square & $-18.00$ & $-30.68$ & $17.13$ & $29.81$ \\
        Binary Regular & $-24.00$ & $-49.25$ & $0.42$ & $25.68$ \\
        Uniform Square & $-24.34$ & $-33.42$ & $13.40$ & $22.48$ \\
        Uniform Regular & $-30.19$ & $-45.02$ & $5.34$ & $20.17$ \\
        Uniform Barabasi-Albert (2) & $-43.71$ & $-78.34$ & $13.49$ & $48.12$ \\
        Binary Barabasi-Albert (2) & $-58.00$ & $-117.83$ & $17.99$ & $77.82$ \\
        Uniform Barabasi-Albert (1) & $-63.36$ & $-84.77$ & $15.19$ & $36.60$ \\
        Binary Barab

In [6]:
import pickle
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx, dimod
from dwave.samplers import SteepestDescentSolver

SD = SteepestDescentSolver()

assert len(graphs) == len(labels) == len(SA_sample_sets) == len(QA_sample_sets)

def logical_bqm(graph):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")
    h = {n: h.get(n, 0.0) for n in graph.nodes}
    return dimod.BinaryQuadraticModel.from_ising(h, J)

def weighted_stats(e, w):
    w = np.asarray(w, dtype=float)
    mean = np.average(e, weights=w)
    return {"best": e.min(), "expected": mean,
            "std": np.sqrt(np.average((e - mean) ** 2, weights=w)),
            "frac_at_best": w[np.isclose(e, e.min())].sum() / w.sum(),
            "distinct": len(e)}

def evaluate(graph, label, SA_ss, QA_ss):
    bqm = logical_bqm(graph)
    rows = []
    for name, ss in (("SA", SA_ss), ("QA", QA_ss)):
        assert set(ss.variables) == set(bqm.variables), f"{label} {name}: variable mismatch"
        occ = ss.record.num_occurrences
        e_raw = bqm.energies(ss)                      # logical objective, not the stored energies

        # one row per read so occurrence counts are preserved through postprocessing
        init = np.repeat(ss.record.sample, occ, axis=0)
        pp = SD.sample(bqm, initial_states=(init, list(ss.variables)))
        e_before = np.repeat(e_raw, occ)
        e_after = pp.record.energy
        assert np.all(e_after <= e_before + 1e-9), "SD output not aligned with initial states"
        flips = (pp.record.sample != init).sum(axis=1)

        extra = {"n_reads": int(occ.sum()),
                 "frac_already_local_min": np.mean(flips == 0),
                 "mean_flips": flips.mean(),
                 "chain_break_frac": (ss.record.chain_break_fraction.mean()
                                      if "chain_break_fraction" in ss.record.dtype.names else np.nan)}
        base = {"label": label, "method": name}
        rows.append({**base, "post": False, **weighted_stats(e_raw, occ), **extra})
        rows.append({**base, "post": True,  **weighted_stats(e_after, np.ones_like(e_after)), **extra})
    return rows

rows = []
for g, lab, sa, qa in zip(graphs, labels, SA_sample_sets, QA_sample_sets):
    rows += evaluate(g, lab, sa, qa)

df = pd.DataFrame(rows)
import matplotlib
matplotlib.use("Agg")  # write files without needing a display
import matplotlib.pyplot as plt


def plot_sd_effect(df, labels, outdir="../plots/", basename="sd_effect",
                   baseline=0.0, format="png", group_size=3):
    """
    df       : DataFrame from the evaluation loop (columns: label, method, post, best, expected)
    labels   : instance labels in the order they should appear on the x axis
    baseline : value where the bars start. 0 by default; set it closer to the data
               (e.g. -700) to magnify small differences.
    Writes <basename>_best, <basename>_expected, and <basename>_combined files.
    Each label gets four bars: SA, SA+SD, QA, QA+SD. The y axis is inverted so that
    more negative (better) energies give taller bars.
    """
    if df["label"].nunique() != len(labels) or set(df["label"]) != set(labels):
        raise ValueError("labels do not match the labels in df")
    if len(set(labels)) != len(labels):
        raise ValueError("labels must be unique")

    idx = df.set_index(["label", "method", "post"])

    def values(col, method, post):
        return np.array([float(idx.loc[(lab, method, post), col]) for lab in labels])

    n = len(labels)
    x = np.arange(n)
    width = 0.2
    # (method, post, legend text, color, hatch, offset in bar widths)
    series = [
        ("SA", False, "SA",              "tab:blue",   None,  -1.5),
        ("SA", True,  "SA + steepest descent", "lightskyblue", "//", -0.5),
        ("QA", False, "QA",              "tab:orange", None,   0.5),
        ("QA", True,  "QA + steepest descent", "navajowhite",  "//",  1.5),
    ]

    def draw(ax, col, title):
        all_vals = []
        for method, post, text, color, hatch, off in series:
            v = values(col, method, post)
            all_vals.append(v)
            ax.bar(x + off * width, v - baseline, width, bottom=baseline, label=text,
                   color=color, hatch=hatch, edgecolor="black", linewidth=0.5)

        # Faint separators between groups
        for b in np.arange(group_size, n, group_size) - 0.5:
            ax.axvline(b, color="gray", alpha=0.35, linewidth=1.0, zorder=0)

        lo = min(v.min() for v in all_vals)
        pad = 0.03 * abs(lo - baseline)
        ax.set_ylim(baseline, lo - pad)          # baseline at the bottom, more negative at the top
        ax.set_xlim(-0.5, n - 0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Energy (higher bar = lower energy)")
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.legend(ncol=2, fontsize=8)

    def save(fig, name):
        fig.savefig(f"{outdir}/{name}.{format}", dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Separate figures: best and expected
    for col, key, title in (("best", "best", "Best (lowest) energy"),
                            ("expected", "expected", "Expected (average) energy")):
        fig, ax = plt.subplots(figsize=(14, 5.5))
        draw(ax, col, f"Effect of steepest descent: {title}")
        fig.tight_layout()
        save(fig, f"{basename}_{key}")

    # Combined figure: both panels stacked
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    draw(axes[0], "best", "Best (lowest) energy")
    draw(axes[1], "expected", "Expected (average) energy")
    fig.suptitle("Effect of steepest descent on SA and QA")
    fig.tight_layout()
    save(fig, f"{basename}_combined")


plot_sd_effect(df, labels)
df["variant"] = df["method"] + np.where(df["post"], "+SD", "")
pd.set_option("display.width", 220, "display.max_columns", 30, "display.float_format", "{:.2f}".format)

# 1. Four-way comparison per graph
for stat in ("best", "expected"):
    print(f"\n=== {stat} energy ===")
    print(df.pivot_table(index="label", columns="variant", values=stat)[["SA", "SA+SD", "QA", "QA+SD"]])

# 2. Gain from postprocessing (positive = lower energy)
g = df.pivot_table(index=["label", "method"], columns="post", values=["best", "expected"])
gain = pd.DataFrame({"best_gain": g[("best", False)] - g[("best", True)],
                     "expected_gain": g[("expected", False)] - g[("expected", True)]})
print("\n=== gain from steepest descent ===")
print(gain)

# 3. Sample-property diagnostics (before postprocessing)
print("\n=== diagnostics ===")
print(df[~df["post"]][["label", "method", "n_reads", "distinct", "std", "frac_at_best",
                       "frac_already_local_min", "mean_flips", "chain_break_frac"]])


=== best energy ===
variant                           SA    SA+SD       QA    QA+SD
label                                                          
Binary Barabasi-Albert (1)  -4159.00 -4159.00 -4087.00 -4141.00
Binary Barabasi-Albert (2)  -1604.00 -1604.00 -1546.00 -1566.00
Binary Erdos-Renyi          -1235.00 -1235.00 -1071.00 -1229.00
Binary Full                 -1666.00 -1666.00 -1502.00 -1666.00
Binary Regular              -1344.00 -1344.00 -1320.00 -1328.00
Binary Square               -3713.00 -3713.00 -3695.00 -3703.00
Normal Barabasi-Albert (1)  -4130.26 -4130.26 -3787.87 -4093.98
Normal Barabasi-Albert (2)  -1519.39 -1519.39 -1338.87 -1470.54
Normal Erdos-Renyi          -1205.84 -1205.84 -1072.18 -1203.81
Normal Full                 -1684.68 -1684.68 -1485.08 -1675.56
Normal Regular              -1311.06 -1311.06 -1200.39 -1278.71
Normal Square               -3525.03 -3525.03 -3285.13 -3449.69
Uniform Barabasi-Albert (1) -2465.49 -2465.49 -2402.14 -2454.44
Uniform Barabasi-Al

In [7]:
def plot_sa_vs_qa(sa_results, qa_results, labels, outdir=".", basename="sa_vs_qa",
                  baseline=0.0, format="png", group_size=3):
    """
    sa_results, qa_results : lists of (best, expected) pairs, same order as labels
    baseline   : value where the bars start. 0 by default; set it closer to the data
                 (e.g. -700) to magnify small differences.
    group_size : a faint vertical separator is drawn after every `group_size` labels
    Writes <basename>_best, <basename>_expected, and <basename>_combined files.
    The y axis is inverted, so more negative (better) results give taller bars.
    """
    if not (len(sa_results) == len(qa_results) == len(labels)):
        raise ValueError(f"Length mismatch: sa={len(sa_results)}, qa={len(qa_results)}, labels={len(labels)}")

    sa = np.array([[float(b), float(e)] for b, e in sa_results])  # columns: best, expected
    qa = np.array([[float(b), float(e)] for b, e in qa_results])

    n = len(labels)
    x = np.arange(n)
    width = 0.38
    colors = {"SA": "tab:blue", "QA": "tab:orange"}

    def draw(ax, col, title):
        # bottom=baseline with height=value-baseline makes each bar span baseline -> value
        ax.bar(x - width / 2, sa[:, col] - baseline, width, bottom=baseline,
               label="Simulated annealing", color=colors["SA"], edgecolor="black", linewidth=0.5)
        ax.bar(x + width / 2, qa[:, col] - baseline, width, bottom=baseline,
               label="Quantum annealing", color=colors["QA"], edgecolor="black", linewidth=0.5)

        # Faint separators between groups (e.g. Uniform / Normal / Binary of one graph type)
        for b in np.arange(group_size, n, group_size) - 0.5:
            ax.axvline(b, color="gray", alpha=0.35, linewidth=1.0, zorder=0)

        lo = min(sa[:, col].min(), qa[:, col].min())
        pad = 0.03 * abs(lo - baseline)
        ax.set_ylim(baseline, lo - pad)          # baseline at the bottom, more negative at the top
        ax.set_xlim(-0.5, n - 0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Energy (higher bar = lower energy)")
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.legend()

    def save(fig, name):
        fig.savefig(f"{outdir}/{name}.{format}", dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Separate figures: best and expected
    for col, key, title in ((0, "best", "Best (lowest) energy"),
                            (1, "expected", "Expected (average) energy")):
        fig, ax = plt.subplots(figsize=(12, 5.5))
        draw(ax, col, f"Simulated vs. quantum annealing: {title}")
        fig.tight_layout()
        save(fig, f"{basename}_{key}")

    # Combined figure: both panels stacked
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    draw(axes[0], 0, "Best (lowest) energy")
    draw(axes[1], 1, "Expected (average) energy")
    fig.suptitle("Simulated vs. quantum annealing")
    fig.tight_layout()
    save(fig, f"{basename}_combined")

plot_sa_vs_qa(sa_results, qa_results, labels, outdir="../plots/")

In [8]:
steepest_descent_solver = SteepestDescentSolver()
SAMPLE_SETS_PER_LABEL = 10
OUTPUT_DIR = Path("../plots")


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------
def load_pickle(file_name):
    raw_bytes = Path(file_name).read_bytes()
    if raw_bytes.startswith(b"version https://git-lfs"):
        raise RuntimeError(f"{file_name} is a Git LFS pointer. Run `git lfs pull`.")
    return pickle.loads(raw_bytes)


# ---------------------------------------------------------------------------
# Energies before and after steepest descent
# ---------------------------------------------------------------------------
def ising_coefficients(graph):
    """Node order, linear biases h, and quadratic biases J from the graph's weight attributes."""
    node_order = list(graph.nodes)
    node_weights = nx.get_node_attributes(graph, "weight")
    linear_biases = {node: node_weights.get(node, 0.0) for node in node_order}
    quadratic_biases = nx.get_edge_attributes(graph, "weight")
    return node_order, linear_biases, quadratic_biases


def stack_reads(sample_sets, node_order):
    """
    Pool sample sets into one spin array (one row per read, columns in node_order)
    plus the stored logical energy of each read. Rows are repeated by num_occurrences.
    """
    spin_rows, stored_energies = [], []
    for sample_set in sample_sets:
        sample_set = sample_set.change_vartype("SPIN", inplace=False)
        variable_list = list(sample_set.variables)
        column_order = [variable_list.index(node) for node in node_order]
        repeats = sample_set.record.num_occurrences
        spin_rows.append(np.repeat(sample_set.record.sample[:, column_order], repeats, axis=0))
        stored_energies.append(np.repeat(sample_set.record.energy, repeats))
    return np.vstack(spin_rows), np.concatenate(stored_energies)


def energies_before_and_after(graph, sample_sets):
    """Per-read energies before (stored) and after steepest descent for one label."""
    node_order, linear_biases, quadratic_biases = ising_coefficients(graph)
    spins, energies_before = stack_reads(sample_sets, node_order)
    descended = steepest_descent_solver.sample_ising(
        linear_biases, quadratic_biases,
        initial_states=(spins, node_order),
        num_reads=len(spins),
    )
    return energies_before, descended.record.energy


# ---------------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------------
def summarize(energies_before_per_group, energies_after_per_group):
    """Best (lowest) and expected (mean) energy per group, before and after steepest descent."""
    def reduce(per_group_energies, reducer):
        return np.array([reducer(energies) for energies in per_group_energies])

    return {
        "best":     (reduce(energies_before_per_group, np.min),  reduce(energies_after_per_group, np.min)),
        "expected": (reduce(energies_before_per_group, np.mean), reduce(energies_after_per_group, np.mean)),
    }


def pool_by_group(group_of_label, labels, energies_before_per_label, energies_after_per_label):
    """
    Pool all reads of every label that shares group_of_label(label).
    Returns (group_names, summary), with groups in first-appearance order.
    """
    pooled_before, pooled_after = {}, {}
    for label, before, after in zip(labels, energies_before_per_label, energies_after_per_label):
        group = group_of_label(label)
        pooled_before.setdefault(group, []).append(before)
        pooled_after.setdefault(group, []).append(after)

    group_names = list(pooled_before)
    summary = summarize([np.concatenate(pooled_before[g]) for g in group_names],
                        [np.concatenate(pooled_after[g]) for g in group_names])
    return group_names, summary


# Labels look like "<weight distribution> <graph type>", e.g. "Normal Barabasi-Albert (1)"
def weight_distribution_of(label):
    return label.split(" ", 1)[0]


def graph_type_of(label):
    return label.split(" ", 1)[1]


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------
METRIC_TITLES = {"best": "Best (lowest) energy", "expected": "Expected (average) energy"}


def draw_before_after_bars(axis, summary, metric, x_names, baseline, separator_every, before_name):
    positions = np.arange(len(x_names))
    bar_width = 0.38
    energies_before, energies_after = summary[metric]

    for offset, energies, name, color, hatch in (
        (-0.5, energies_before, before_name,              "tab:orange",  None),
        (+0.5, energies_after,  "QA + steepest descent",  "navajowhite", "//"),
    ):
        axis.bar(positions + offset * bar_width, energies - baseline, bar_width, bottom=baseline,
                 label=name, color=color, hatch=hatch, edgecolor="black", linewidth=0.5)

    if separator_every:
        for boundary in np.arange(separator_every, len(x_names), separator_every) - 0.5:
            axis.axvline(boundary, color="gray", alpha=0.35, zorder=0)

    lowest_energy = min(energies_before.min(), energies_after.min())
    axis.set_ylim(baseline, lowest_energy - 0.03 * abs(lowest_energy - baseline))  # more negative at top
    axis.set_xlim(-0.5, len(x_names) - 0.5)
    axis.set_xticks(positions)
    rotation = 45 if len(x_names) > 4 else 0
    axis.set_xticklabels(x_names, rotation=rotation, ha="right" if rotation else "center", fontsize=8)
    axis.set_ylabel("Energy (higher bar = lower energy)")
    axis.set_title(METRIC_TITLES[metric])
    axis.grid(axis="y", linestyle="--", alpha=0.4)
    axis.set_axisbelow(True)
    axis.legend(fontsize=8)


def save_before_after_charts(summary, x_names, file_prefix, baseline=0.0,
                             separator_every=None, before_name="QA (10x reads)"):
    """Writes <file_prefix>_best.png, _expected.png, and _combined.png into OUTPUT_DIR."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    options = dict(summary=summary, x_names=x_names, baseline=baseline,
                   separator_every=separator_every, before_name=before_name)

    for metric in METRIC_TITLES:
        figure, axis = plt.subplots(figsize=(12, 5.5))
        draw_before_after_bars(axis, metric=metric, **options)
        figure.tight_layout()
        figure.savefig(OUTPUT_DIR / f"{file_prefix}_{metric}.png", dpi=300)
        plt.close(figure)

    figure, axes = plt.subplots(2, 1, figsize=(12, 10))
    for axis, metric in zip(axes, METRIC_TITLES):
        draw_before_after_bars(axis, metric=metric, **options)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"{file_prefix}_combined.png", dpi=300)
    plt.close(figure)


assert len(graphs) == len(labels)
assert len(pt4_results) == SAMPLE_SETS_PER_LABEL * len(labels)

energies_before_per_label, energies_after_per_label = [], []
for label_index, graph in enumerate(graphs):
    first = label_index * SAMPLE_SETS_PER_LABEL
    before, after = energies_before_and_after(graph, pt4_results[first:first + SAMPLE_SETS_PER_LABEL])
    energies_before_per_label.append(before)
    energies_after_per_label.append(after)

# 1. One pair of bars per label (separator every 3 labels)
per_label_summary = summarize(energies_before_per_label, energies_after_per_label)
save_before_after_charts(per_label_summary, labels, "qa10x_sd_effect", separator_every=3)

# 2. Pooled by graph type (Uniform, Normal, Binary together)
graph_types, by_graph_type = pool_by_group(graph_type_of, labels,
                                           energies_before_per_label, energies_after_per_label)
save_before_after_charts(by_graph_type, graph_types, "qa10x_by_graph_type",
                         before_name="QA (10x reads, pooled)")

# 3. Pooled by weight distribution (all graph types together)
distributions, by_distribution = pool_by_group(weight_distribution_of, labels,
                                               energies_before_per_label, energies_after_per_label)
save_before_after_charts(by_distribution, distributions, "qa10x_by_weight_dist",
                         before_name="QA (10x reads, pooled)")

In [9]:
OUTPUT_DIR = Path("../plots")


def draw_best_vs_expected(axis, results, labels, method_name, color,
                          baseline=0.0, separator_every=3):
    """
    One method on one axis: a best bar and an expected bar for every label.
    results : list of (best, expected) pairs in the same order as labels.
    """
    best_energies = np.array([float(best) for best, _ in results])
    expected_energies = np.array([float(expected) for _, expected in results])

    positions = np.arange(len(labels))
    bar_width = 0.38

    for offset, energies, name, bar_color, hatch in (
        (-0.5, best_energies,     f"{method_name} best",     color,       None),
        (+0.5, expected_energies, f"{method_name} expected", color,       "//"),
    ):
        axis.bar(positions + offset * bar_width, energies - baseline, bar_width, bottom=baseline,
                 label=name, color=bar_color, alpha=1.0 if hatch is None else 0.55,
                 hatch=hatch, edgecolor="black", linewidth=0.5)

    if separator_every:
        for boundary in np.arange(separator_every, len(labels), separator_every) - 0.5:
            axis.axvline(boundary, color="gray", alpha=0.35, zorder=0)

    lowest_energy = min(best_energies.min(), expected_energies.min())
    axis.set_ylim(baseline, lowest_energy - 0.03 * abs(lowest_energy - baseline))  # more negative at top
    axis.set_xlim(-0.5, len(labels) - 0.5)
    axis.set_xticks(positions)
    axis.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    axis.set_ylabel("Energy (higher bar = lower energy)")
    axis.set_title(f"{method_name}: best vs. expected energy")
    axis.grid(axis="y", linestyle="--", alpha=0.4)
    axis.set_axisbelow(True)
    axis.legend(fontsize=8)


def save_best_vs_expected_charts(sa_results, qa_results, labels,
                                 baseline=0.0, separator_every=3, file_prefix="best_vs_expected"):
    """
    Writes <file_prefix>_sa.png, <file_prefix>_qa.png, and <file_prefix>_combined.png
    (SA and QA as two separate panels, each with its own y axis) into OUTPUT_DIR.
    """
    if not (len(sa_results) == len(qa_results) == len(labels)):
        raise ValueError(f"Length mismatch: sa={len(sa_results)}, qa={len(qa_results)}, labels={len(labels)}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    methods = (("SA", sa_results, "tab:blue"), ("QA", qa_results, "tab:orange"))
    options = dict(labels=labels, baseline=baseline, separator_every=separator_every)

    for method_name, results, color in methods:
        figure, axis = plt.subplots(figsize=(12, 5.5))
        draw_best_vs_expected(axis, results, method_name=method_name, color=color, **options)
        figure.tight_layout()
        figure.savefig(OUTPUT_DIR / f"{file_prefix}_{method_name.lower()}.png", dpi=300)
        plt.close(figure)

    figure, axes = plt.subplots(2, 1, figsize=(12, 10))
    for axis, (method_name, results, color) in zip(axes, methods):
        draw_best_vs_expected(axis, results, method_name=method_name, color=color, **options)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"{file_prefix}_combined.png", dpi=300)
    plt.close(figure)


save_best_vs_expected_charts(sa_results, qa_results, labels)